In [0]:
spark.sql("""create Volume if not exists group_project.bronze.autoloader_metadata""")
# here i creatd the folder for auto loader to track the checkpoints

In [0]:
from common_utils.logging import get_logger

logger = get_logger("Auto_loader")

# =========================================
# 1. Source and metadata paths
# =========================================

source_dir = "/Volumes/group_project/bronze/raw_files"

metadata_dir = "/Volumes/group_project/bronze/autoloader_metadata"


# =========================================
# 2. Configuration
# =========================================

config = {
    "source" : {
        "DimCancellationReason*.csv": "dim_cancel",
        "DimCounty*.csv": "dim_county",
        "DimPeople*.csv": "dim_people",
        "DimShrinkageType*.csv": "dim_shrinkage",
        "FactWork*.csv": "factwork",
        "FactShift*.csv": "factshift",
        "FactShrinkage*.csv": "factshrinkage"
    }
}

# =========================================
# 3. Auto Loader
# =========================================

for file_pattern, table_name in config["source"].items():
    source_path = source_dir
    target_table = f"group_project.bronze.{table_name}"
    schema_path = f"{metadata_dir}/schema/{table_name}"

    checkpoint_path = f"{metadata_dir}/checkpoint/{table_name}"

    logger.info("Starting ingestion for %s", table_name)

    # reading using autoloader
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation",schema_path)
        .option("cloudFiles.includeExistingFiles","true")
        .option("header","true")
        .option("pathGlobFilter", file_pattern)
        .load(source_path)
    )
    # Write into Bronze Delta table
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    # Wait for current ingestion to finish
    query.awaitTermination()
    logger.info("Successfully ingested %s", target_table)


logger.info("All Bronze ingestion completed successfully")



